In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [3]:
model = ChatOllama(model= "llama3.2")

In [4]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detail feedback for the essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)

In [5]:
structured_model =model.with_structured_output(EvaluationSchema)

In [6]:
essay = """Artificial Intelligence (AI) is one of the most important technological developments in the modern world. It is transforming various fields, including healthcare, business, communication, and education. AI has changed the way students learn and teachers teach. With the help of AI tools such as ChatGPT, virtual tutors, and smart learning platforms, education is becoming more accessible, interactive, and effective.

One of the major impacts of AI in education is personalized learning. Every student has a different learning style and speed. AI helps students learn according to their individual needs and abilities. It can explain difficult topics, provide practice questions, and suggest learning materials. Students can also use AI tools to solve doubts and improve their understanding of different subjects.

AI is also helpful for teachers. It can assist them in preparing lesson plans, creating assignments, checking objective questions, and organizing educational materials. This reduces their workload and allows them to spend more time guiding students. Furthermore, AI-powered tools such as language translators, voice assistants, and speech-to-text applications make education more accessible to students with different needs.

However, AI also has some negative impacts on education. Students may become too dependent on AI and stop thinking independently. Some students use AI to complete assignments without understanding the concepts. Moreover, AI can sometimes provide incorrect information, which may confuse learners. Privacy, unequal access to technology, and the cost of certain AI tools are also important concerns.

To overcome these challenges, AI should be used responsibly. Students should use it as a learning assistant rather than a shortcut to completing their work. Teachers and educational institutions should teach students how to verify information, protect personal data, and use AI ethically. Human interaction, creativity, and critical thinking should remain essential parts of education.

In conclusion, Artificial Intelligence is changing the education system in many positive ways. It makes learning more personalized, saves time, and provides access to a wide range of educational resources. Although it has certain disadvantages, its benefits can be increased through proper guidance and responsible use. AI should not replace teachers; instead, it should work alongside them to create a better learning environment. When used wisely, AI can play a significant role in improving education and preparing students for the future."""

In [8]:
prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}"

In [9]:
structured_model.invoke(prompt)

EvaluationSchema(feedback='The essay provides a clear and comprehensive overview of the impact of Artificial Intelligence (AI) on education. The writer effectively highlights both the benefits and drawbacks of AI in education, offering practical suggestions for responsible use. However, the essay could benefit from more nuanced analysis, additional examples, and varied sentence structures to improve its overall flow and impact.', score=8)

In [37]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    overall_feedback: str
    average_score: float

In [ ]:
def evaluate_language(state: UPSCState):
    prompt = f"""
    Evaluate the language quality of the following essay.
    Provide constructive feedback and assign a score out of 10.

    Essay:
    {state['essay']}
    """

    output = structured_model.invoke(prompt)

    return {
        "language_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [39]:
def evaluate_analysis(state: UPSCState):
    prompt = f"""
    Evaluate the depth of analysis of this essay.
    Give feedback and a score out of 10.

    Essay: {state['essay']}
    """
    output = structured_model.invoke(prompt)

    return {
        "analysis_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [41]:
def evaluate_thought(state: UPSCState):
    prompt = f"""
    Evaluate the clarity of thought of this essay.
    Give feedback and a score out of 10.

    Essay: {state['essay']}
    """
    output = structured_model.invoke(prompt)

    return {
        "clarity_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [42]:
def final_evaluation(state: UPSCState):
    scores = state.get("individual_scores", [])

    if not scores:
        raise ValueError(
            "No evaluation scores found. Check that all evaluation nodes run first."
        )

    average_score = sum(scores) / len(scores)

    prompt = f"""
    Provide an overall evaluation of the essay.

    Language Feedback: {state['language_feedback']}
    Analysis Feedback: {state['analysis_feedback']}
    Clarity Feedback: {state['clarity_feedback']}

    Individual Scores: {scores}
    Average Score: {average_score:.2f}/10

    Give a concise overall assessment and suggestions for improvement.
    """

    final_feedback = model.invoke(prompt).content

    return {
        "overall_feedback": final_feedback,
        "average_score": average_score
    }

In [43]:
graph = StateGraph(UPSCState)

graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

#edges 
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")

graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_analysis", "final_evaluation")
graph.add_edge("evaluate_thought", "final_evaluation")

graph.add_edge("final_evaluation", END)

workflow = graph.compile()

In [44]:
inital_state = {
    "essay": essay
}
workflow.invoke(inital_state)

{'essay': 'Artificial Intelligence (AI) is one of the most important technological developments in the modern world. It is transforming various fields, including healthcare, business, communication, and education. AI has changed the way students learn and teachers teach. With the help of AI tools such as ChatGPT, virtual tutors, and smart learning platforms, education is becoming more accessible, interactive, and effective.\n\nOne of the major impacts of AI in education is personalized learning. Every student has a different learning style and speed. AI helps students learn according to their individual needs and abilities. It can explain difficult topics, provide practice questions, and suggest learning materials. Students can also use AI tools to solve doubts and improve their understanding of different subjects.\n\nAI is also helpful for teachers. It can assist them in preparing lesson plans, creating assignments, checking objective questions, and organizing educational materials. T